# Capstone: Build an Instruction-Following Assistant (T2, one free T4)

Qwen3-1.7B-Base -> data card -> SFT (QLoRA) -> on-policy pairs -> DPO (QLoRA) -> IFEval, judge win rate, XSTest -> model card.

Run the cells from the top. Each stage skips work that is already done, so after a timeout reconnect, re-run the setup cells, and run the stage again: it resumes from its newest checkpoint. The page's runbook gives the time each stage takes.

In [ ]:
import os, sys, shutil, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
assert sys.version_info >= (3, 11), 'Python 3.11 or newer is needed: pick a newer runtime image'
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    raise SystemExit('No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2, Internet on.')
print('free disk GB', shutil.disk_usage('/').free // 2**30)

In [ ]:
# Pinned versions (checked 2026-11-10). Kaggle's torchao conflicts with PEFT, so it goes first.
!pip uninstall -y -q torchao
!pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 bitsandbytes==0.50.2 lm-eval==0.4.13 langdetect immutabledict nltk
import nltk; [nltk.download(p, quiet=True) for p in ('punkt', 'punkt_tab')]

In [ ]:
# The code: the same package the reference run used
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/genieincodebottle/aiml-companion.git
!cd aiml-companion && git sparse-checkout set courses/llm-training/labs/lltCapstoneAssistant
%cd aiml-companion/courses/llm-training/labs/lltCapstoneAssistant
!python tests/test_cpu.py

In [ ]:
# Where the work lives. Colab: mount Drive so a new session finds it.
if 'google.colab' in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/lltCapstoneAssistant/work'
else:
    WORK = '/kaggle/working/work'      # attach the previous version's output and copy it here to resume
os.makedirs(WORK, exist_ok=True)
A = f'--tier T2 --work {WORK}'

## Stage 1. Data and the data card (CPU, downloads about 3 GB)

In [ ]:
!python -m capstone.data $A
!cat {WORK}/data/card.json

## Stage 2. SFT with QLoRA

The longest stage. It saves a checkpoint every 50 steps. To skip it, run `!python -m capstone.fetch --what sft $A` instead: it merges our SFT adapter into the base.

In [ ]:
!python -m capstone.sft $A

## Stage 3. On-policy samples, then reward-model labels and pairs

In [ ]:
!GEN_BATCH=32 python -m capstone.sample $A
!python -m capstone.label $A

## Stage 4. DPO with QLoRA

In [ ]:
!python -m capstone.dpo $A

## Stage 5. Evaluation generations, IFEval and XSTest scores, then the judge

The T2 judge is Olmo-3-7B-Instruct in 4-bit. Our run checked it against HelpSteer3 human labels; the page gives its agreement next to the T3 judge's. On a T4 the default judge cell runs three comparisons and takes about 7 hours (calibration 34 min, then 129 min per comparison). Add `--pairs dpo:official` to run one comparison in about 163 minutes.

In [ ]:
!GEN_BATCH=16 python -m capstone.evalgen $A
!python -m capstone.scores $A
!GEN_BATCH=4 python -m capstone.judge $A
# a second refusal rule, checked against XSTest's human labels first
!GEN_BATCH=8 python -m capstone.refusal $A

## Stage 5d. Label 40 of your own comparisons by hand

Export, then open `handlabel/sheet.jsonl`, read each request and both replies, and write A, B or C in `label` before you look at any judge verdict. Then score.

In [ ]:
!python -m capstone.handlabel export $A --n 40
# ... fill the labels, then:
# !python -m capstone.handlabel score $A

## Stage 6. Model card, then release to your own Hub account

Write `limitations.md` from your results first.

In [ ]:
!python -m capstone.card $A --limitations limitations.md
# from huggingface_hub import HfApi; HfApi().upload_folder(folder_path=f'{WORK}/dpo/merged', repo_id='<you>/<name>')